# Inflation-linked bonds

**Start here:** This deep dive expands on `02_pricing/pricing_across_asset_classes.ipynb`; use `02_pricing/pricing_fundamentals.ipynb` for the shared instrument JSON -> market -> model pipeline.

**Prerequisites:** `02_pricing/pricing_fundamentals.ipynb`. **`inflation_linked_bond`** needs a **real discount** curve plus a typed **`InflationIndex`** in the `MarketContext`.


## Concept

- **Real coupon** plus **index ratio** from CPI fixes cashflows.
- **Real yield** and **breakeven inflation** are interpreted relative to nominal curves (this notebook prints registry metrics when available).
- The typed inflation index uses the Rust default of no lag because the bond applies its own lag (avoids double-lagging).


In [1]:
import json
from datetime import date

import sys
sys.path.insert(0, "../..")

from _shared import DEMO_AS_OF, print_metrics
from finstack_quant.valuations import ValuationResult
from finstack_quant.valuations.instruments import price_instrument_with_metrics, validate_instrument_json
from finstack_quant.core.market_data import DiscountCurve, InflationIndex, MarketContext

print("Imports OK (inflation-linked).")


Imports OK (inflation-linked).


## Instrument JSON


In [2]:
AS_OF = DEMO_AS_OF
AS_OF_STR = AS_OF.isoformat()

ilb = json.dumps(
    {
        "type": "inflation_linked_bond",
        "spec": {
            "id": "TIPS-DEMO",
            "notional": {"amount": "1000000", "currency": "USD"},
            "issue_date": "2024-01-15",
            "maturity": "2034-01-15",
            "real_coupon": "0.025",
            "frequency": {"count": 6, "unit": "months"},
            "day_count": "Thirty360",
            "bdc": "following",
            "calendar_id": "weekends_only",
            "stub": "None",
            "base_date": "2024-01-15",
            # Matches synthetic US-CPI linear series at issue (see MarketContext cell; ~277.07 at 2024-01-15).
            "base_index": 277.0676975464149,
            "indexation_method": "TIPS",
            "inflation_index_id": "US-CPI",
            "lag": {"months": 3},
            "deflation_protection": "MaturityOnly",
            "discount_curve_id": "USD-TIPS",
            "quoted_clean": 100.0,
            "attributes": {"tags": [], "meta": {}},
            "pricing_overrides": {},
        },
    }
)

validate_instrument_json(ilb)
print("ILB JSON OK")


ILB JSON OK


## MarketContext


In [3]:
tips = DiscountCurve(
    "USD-TIPS",
    AS_OF,
    [(0.0, 1.0), (1.0, 0.97), (5.0, 0.85), (10.0, 0.70)],
    day_count="act_365f",
)
obs = []
y, m, v = 2019, 10, 250.0
for _ in range(80):
    obs.append((date(y, m, 1), v))
    m += 1
    if m > 12:
        m, y = 1, y + 1
    v *= 1.002
inflation_index = InflationIndex("US-CPI", obs, "USD", interpolation="linear")
mc = MarketContext().insert(tips)
mc.insert_inflation_index(inflation_index)
market_json = mc.to_json()
print("inflation index observations:", len(mc.get_inflation_index("US-CPI")))


inflation index observations: 80


## Pricing


In [4]:
out = price_instrument_with_metrics(ilb, market_json, AS_OF_STR, model="discounting")
print(ValuationResult.from_json(out))


ValuationResult(id="TIPS-DEMO", price=973849.5729, currency=USD, metrics=0)


## Metrics


In [5]:
metrics = ["breakeven_inflation", "index_ratio"]
out = price_instrument_with_metrics(
    ilb,
    market_json,
    AS_OF_STR,
    model="discounting",
    metrics=metrics,
)
print_metrics(ValuationResult.from_json(out), metrics)
print("inflation metrics pass complete")


  breakeven_inflation: 0.010270027195817375
  index_ratio: 1.0181446740200375
inflation metrics pass complete


## Takeaways

- **Inflation indices** ride alongside curves in the v2 market snapshot.
- **Breakeven** and **index ratio** metrics connect linker pricing to macro intuition.
- Match **lag policy** between bond spec and index payload to avoid double-lag validation errors.
